In [43]:
import numpy as np
import torch
import gymnasium as gym
import gymnasium.spaces as spaces
import random

class APTPOMDPEnv(gym.Env):
    def __init__(self, attack_sequences):
        # 状态空间：隐藏的真实攻击类型
        self.true_state_space = spaces.Discrete(5)  # 5种攻击技术
        
        # 观测空间：可见的攻击特征
        self.observation_space = spaces.Box(low=0, high=1, shape=(5,))
        
        # 动作空间：边操作（添加/删除/保持）+ 目标节点对
        self.action_space = spaces.Box(low=0, high=1, shape=(3, 5, 5))  # [操作类型, 源节点, 目标节点]
        
        # 攻击序列数据集
        self.attack_sequences = attack_sequences
        self.current_seq = None
        self.current_step = 0
        
        # 因果图存储
        self.causal_graph = np.zeros((5, 5))
        
        # 技术映射
        self.tech_map = {
            "T1595_扫描侦查": 0,
            "T1190_漏洞利用": 1,
            "T1193_恶意软件投递": 2, 
            "T1598_钓鱼攻击": 3,
            "T1105_C2通信": 4
        }
        
    def _get_obs(self):
        """ 根据真实状态生成部分观测 """
        # 随机隐藏30%的状态信息
        obs = np.zeros(5)
        if self.current_seq is not None:
            tech = self.current_seq[self.current_step]["technique"]
            tech_id = self.tech_map[tech]
            obs[tech_id] = 1
            obs_mask = np.random.choice([0,1], size=5, p=[0.3, 0.7])
            obs = obs * obs_mask
        return obs
        
    def _sample_sequence(self):
        """ 从攻击序列数据集中随机采样一个序列 """
        return random.choice(self.attack_sequences)["sequence"]
    
    def reset(self, seed=None, options=None):
        """ 重置环境并返回初始观测 """
        super().reset(seed=seed)
        self.current_seq = self._sample_sequence()
        self.current_step = 0
        self.causal_graph = np.zeros((5, 5))
        return self._get_obs(), {}
    
    def step(self, action):
        """ 执行动作并返回新状态 """
        op_type, src, dst = action
        
        # 更新因果图
        if op_type[0] > 0.5:    # 添加边
            self.causal_graph[src.argmax()][dst.argmax()] = 1
        elif op_type[1] > 0.5:  # 删除边
            self.causal_graph[src.argmax()][dst.argmax()] = 0
            
        # 环境推进到下一步
        self.current_step += 1
        done = self.current_step >= len(self.current_seq)
        
        # 计算奖励
        reward = self._calculate_reward()
        
        # 生成新观测
        obs = self._get_obs()
        
        return obs, reward, done, False, {"graph": self.causal_graph.copy()}
    
    def _calculate_reward(self):
        """ 奖励函数设计 """
        if self.current_step < 2:
            return 0
            
        # 检查当前步骤与前一步骤是否存在因果关系
        prev_tech = self.current_seq[self.current_step-1]["technique"] 
        curr_tech = self.current_seq[self.current_step]["technique"]
        prev_id = self.tech_map[prev_tech]
        curr_id = self.tech_map[curr_tech]
        
        # 如果存在时序关系的边得到正奖励
        edge_reward = 1.0 if self.causal_graph[prev_id][curr_id] > 0 else -0.1
        
        # 稀疏性惩罚
        sparsity = -0.1 * np.sum(self.causal_graph)
        
        return edge_reward + sparsity

In [51]:
import torch.nn as nn
import torch.nn.functional as F

class GraphConv(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # 定义可学习的权重矩阵
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        
        # 初始化参数
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)
        
    def forward(self, x):
        # x的形状: (batch_size, num_nodes, in_features)
        batch_size = x.size(0)
        num_nodes = x.size(1)
        
        # 计算邻接矩阵的度矩阵
        adj = torch.ones(batch_size, num_nodes, num_nodes, device=x.device)
        deg = adj.sum(dim=-1, keepdim=True).clamp(min=1)
        norm_adj = adj / deg
        
        # 图卷积操作
        support = torch.matmul(x, self.weight)  # (batch_size, num_nodes, out_features)
        output = torch.matmul(norm_adj, support)  # (batch_size, num_nodes, out_features)
        output = output + self.bias
        
        return F.relu(output)

In [52]:
import torch
import torch.nn as nn
class CausalPPOPolicy(nn.Module):
    def __init__(self, observation_space, action_space, lr_schedule, **kwargs):
        super().__init__()
        obs_dim = observation_space.shape[0]
        
        # 动作空间维度检查
        if len(action_space.shape) != 3:
            raise ValueError("动作空间必须是3维的 (例如: [3, 12, 12])")
        self.action_dim = action_space.shape
        
        # 历史编码器
        self.lstm = nn.LSTM(obs_dim, 128, batch_first=True)
        
        # 图状态编码器
        self.gcn = GraphConv(12, 64)
        
        # 设置特征维度
        self.features_dim = 128 + 64 * 12  # 修改特征维度以匹配GCN输出
        
        # 策略头
        total_action_dim = self.action_dim[0] * self.action_dim[1] * self.action_dim[2]
        self.actor = nn.Sequential(
            nn.Linear(self.features_dim, 256),
            nn.ReLU(),
            nn.Linear(256, total_action_dim)
        )
        
        # 价值头
        self.critic = nn.Sequential(
            nn.Linear(self.features_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )
        
    def forward(self, obs):
        # 从obs中提取时序观测和图状态
        obs_seq = obs["obs_seq"] if isinstance(obs, dict) else obs
        graph_state = obs["graph_state"] if isinstance(obs, dict) else None
        
        if graph_state is None:
            # 如果没有图状态,创建一个全零的图状态
            batch_size = obs_seq.size(0)
            graph_state = torch.zeros(batch_size, 12, 12, device=obs_seq.device)
            
        # 处理时序观测 - 确保输入是3维的 [batch, seq_len, features]
        if len(obs_seq.shape) == 2:
            obs_seq = obs_seq.unsqueeze(0)  # 添加batch维度
            
        lstm_out, _ = self.lstm(obs_seq)
        temporal_feat = lstm_out[:, -1, :]  # [batch, 128]
        
        # 处理图状态 - 确保输入是3维的 [batch, nodes, features] 
        if len(graph_state.shape) == 2:
            graph_state = graph_state.unsqueeze(0)  # 添加batch维度
            
        graph_feat = self.gcn(graph_state)  # [batch, 12, 64]
        graph_feat = graph_feat.reshape(graph_feat.size(0), -1)  # [batch, 12*64]
        
        # 联合特征
        joint_feat = torch.cat([temporal_feat, graph_feat], dim=1)
        
        # 动作分布
        action_logits = self.actor(joint_feat).view(-1, *self.action_dim)
        value = self.critic(joint_feat)
        
        return action_logits, value

In [54]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
import json
import numpy as np
import random

def generate_apt_sequence():
    """生成单个APT攻击序列"""
    techniques = [
        "T1595_扫描侦查",
        "T1190_漏洞利用", 
        "T1193_恶意软件投递",
        "T1598_钓鱼攻击",
        "T1105_C2通信"
    ]
    
    sequence = []
    length = random.randint(3, 8)
    
    for _ in range(length):
        tech = random.choice(techniques)
        sequence.append({
            "technique": tech,
            "timestamp": random.randint(1000000, 9999999)
        })
    
    return sequence

def load_apt_sequences(num_sequences=100):
    """生成模拟的APT攻击序列数据集"""
    sequences = []
    for _ in range(num_sequences):
        sequence = generate_apt_sequence()
        sequences.append({
            "id": f"APT{_:03d}",
            "group": f"Group{random.randint(1,10)}",
            "sequence": sequence
        })
    
    return sequences

class GraphLogger(BaseCallback):
    def _on_step(self):
        # 记录因果图演化过程
        graphs = self.locals["infos"]["graph"]
        self.logger.record("graph/density", np.mean(graphs))

# 初始化环境
env = APTPOMDPEnv(load_apt_sequences())
env = Monitor(env)
env = DummyVecEnv([lambda: env])

# 配置自定义策略参数
policy_kwargs = dict(
    features_extractor_class=CausalPPOPolicy,
    features_extractor_kwargs=dict(
        action_space=env.action_space,
        lr_schedule=lambda _: 3e-4
    ),
    net_arch=dict(pi=[256, 256], vf=[256, 256])
)

# 创建PPO智能体
model = PPO(
    "MlpPolicy",
    env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    n_steps=2048,
    batch_size=256,
    learning_rate=3e-4,
    device="cpu",
    tensorboard_log="./causal_log/",
    seed=42  # 添加固定的随机种子
)

# 训练
model.learn(
    total_timesteps=1e6,
    callback=[GraphLogger()],
    tb_log_name="apt_causal"
)

Using cpu device


KeyboardInterrupt: 